# Sudoku Solver 🔢
Sudoku is a **constraint satisfaction problem**: 81 variables, domains 1–9, and "all different" constraints on rows, columns and boxes. Solve it with backtracking search plus the **MRV heuristic** (fill the cell with the fewest options first) and count how much smarter that is than going left to right. This is search, the "good old-fashioned AI" that still runs inside planners and SAT solvers.

Background: [[Calculus and Optimization]] (search vs. optimisation), [[Probabilistic Graphical Models]] (constraints as factors) · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import time, numpy as np
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
parse = lambda s: np.array([0 if ch in ".0" else int(ch) for ch in s]).reshape(9, 9)
EASY = parse("003020600900305001001806400008102900700000008006708200002609500800203009005010300")
HARD = parse("4.....8.5.3..........7......2.....6.....8.4......1.......6.3.7.5..2.....1.4......")
def show(g): print("\n".join(" ".join(str(v) if v else "." for v in row) for row in g))
def valid_solution(g): return all(sorted(g[i]) == list(range(1, 10)) and sorted(g[:, i]) == list(range(1, 10)) for i in range(9)) and \
    all(sorted(g[r:r + 3, c:c + 3].ravel()) == list(range(1, 10)) for r in (0, 3, 6) for c in (0, 3, 6))
show(EASY)

## 1. Candidates
Return the values 1–9 not yet used in cell (r, c)'s row, column or 3×3 box.

In [ ]:
def candidates(grid, r, c):
    # TODO 1: the legal values for an empty cell
    raise NotImplementedError  # TODO 1

In [ ]:
check("candidates", lambda: candidates(EASY, 0, 0) == [4, 5] and candidates(EASY, 4, 4) == [3, 4, 5, 6, 9], "Cell (0,0) of EASY allows [4, 5]; cell (4,4) allows [3, 4, 5, 6, 9].")

## 2. Backtracking with MRV
Recursive: if no empty cell is left, done. Otherwise pick the empty cell with the **fewest** candidates, try each value, recurse, and undo on failure. Increment `stats['nodes']` once per call. Solve **in place** and return `True`/`False`.

In [ ]:
def solve(grid, stats):
    # TODO 2: MRV cell choice + backtracking
    raise NotImplementedError  # TODO 2

In [ ]:
class GaveUp(Exception): pass
def solve_naive(grid, stats, limit=200_000):   # left-to-right, for comparison (gives up after `limit` nodes)
    stats["nodes"] += 1
    if stats["nodes"] > limit: raise GaveUp
    for r in range(9):
        for c in range(9):
            if grid[r, c] == 0:
                for v in candidates(grid, r, c):
                    grid[r, c] = v
                    if solve_naive(grid, stats): return True
                grid[r, c] = 0; return False
    return True

if ready("candidates"):
    try:
        out = {}
        for name, puzzle in [("easy", EASY), ("hard", HARD)]:
            for label, fn in [("naive", solve_naive), ("MRV", solve)]:
                g, st, t = puzzle.copy(), {"nodes": 0}, time.time()
                try: ok = fn(g, st) and valid_solution(g)
                except GaveUp: ok = False
                out[(name, label)] = (ok, st["nodes"])
                print(f"{name:>4} {label:>5}: solved={ok}  {st['nodes']:>7,} search nodes  {time.time()-t:.2f}s")
        check("MRV solves faster", lambda: out[("hard", "MRV")][0] and out[("hard", "MRV")][1] < out[("hard", "naive")][1] / 5,   # naive gives up at 200k nodes (it needs ~9.7M)
              "MRV must solve HARD with at least 5× fewer search nodes than left-to-right.")
    except NotImplementedError:
        print("❌ MRV solves faster: not implemented yet – fill in the TODO above.")

<details><summary>▶ Solution</summary>

```python
def candidates(grid, r, c):
    used = set(grid[r]) | set(grid[:, c]) | set(grid[r // 3 * 3:r // 3 * 3 + 3, c // 3 * 3:c // 3 * 3 + 3].ravel())
    return [v for v in range(1, 10) if v not in used]
```

```python
def solve(grid, stats):
    stats["nodes"] += 1
    empty = [(r, c) for r in range(9) for c in range(9) if grid[r, c] == 0]
    if not empty:
        return True
    r, c = min(empty, key=lambda rc: len(candidates(grid, *rc)))
    for v in candidates(grid, r, c):
        grid[r, c] = v
        if solve(grid, stats):
            return True
    grid[r, c] = 0
    return False
```
</details>

## Stretch goals
- Add **constraint propagation**: after each assignment, fill every cell that has only one candidate left. How many puzzles need no search at all?
- Encode Sudoku as SAT (729 boolean variables) and hand it to a SAT solver.
- Generate puzzles: start from a solved grid and remove clues while the solution stays unique.